# Semana 3 — LAB estructural

Notebook principal para ejecutar y revisar las partes A–D: carga viva, sismo pseudoestático, superposición y capacidad HA.

Los scripts de cálculo se mantienen como módulos reutilizables; este notebook organiza su ejecución y muestra las verificaciones.

## 0. Preparacion

**Archivo:** esta celda pertenece al notebook `Semana3_LAB.ipynb`.

**En palabras simples:** se importan las herramientas, se localiza la carpeta del proyecto y se define donde estan los resultados.

In [ ]:
from pathlib import Path
import json, csv, subprocess, sys
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'results'
print('Carpeta de trabajo:', ROOT)


## 1. Parametros y seccion Fiber

**Archivo:** `P1L3/parametros.json`.

**En palabras simples:** se leen las dimensiones, materiales y armadura que usara el calculo de la columna.

In [ ]:
cfg = json.loads((ROOT / 'parametros.json').read_text(encoding='utf-8'))
col = cfg['columna']
for k in ['b_m','h_m','recubrimiento_hasta_estribo_m','recubrimiento_al_centro_barra_m','diametro_m','estribo_diametro_m','estribo_spacing_m','fc_MPa','fy_MPa']:
    print(f'{k}: {col[k]}')
print('Armadura longitudinal: 16 barras Ø22')


**Archivo:** `P1L3/capacidad.py`, funcion `fibers()`, cuyos resultados se guardan en `P1L3/results/fibras.csv`.

**En palabras simples:** se cuentan y dibujan las pequenas piezas de hormigon y las barras de acero que forman la seccion.

In [ ]:
# Discretización Fiber guardada por el cálculo
rows = list(csv.DictReader((OUT/'fibras.csv').open(encoding='utf-8-sig')))
concrete = [r for r in rows if int(float(r['material'])) == 1]
steel = [r for r in rows if int(float(r['material'])) == 2]
print(f'Fibras de hormigón: {len(concrete)}')
print(f'Fibras de acero: {len(steel)}')
fig, ax = plt.subplots(figsize=(5,5))
ax.scatter([float(r['z_m']) for r in concrete],[float(r['y_m']) for r in concrete],s=2,label='Hormigón')
ax.scatter([float(r['z_m']) for r in steel],[float(r['y_m']) for r in steel],s=35,label='Acero')
ax.set_aspect('equal'); ax.set_xlabel('z [m]'); ax.set_ylabel('y [m]'); ax.legend(); ax.grid(alpha=.2)
plt.show()


## 2. Ejecucion de los casos base

**Archivo:** `P1L3/ejecutar.py`, funcion principal de ejecucion.

**En palabras simples:** se corre todo el modelo, se resuelven `G`, `Q`, `EX`, `EY` y `R`, y se generan los archivos de resultados.

In [ ]:
# Ejecuta G, Q, EX, EY, R y las verificaciones de masa/capacidad.
result = subprocess.run([sys.executable, str(ROOT/'ejecutar.py')], cwd=ROOT, text=True, capture_output=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError('La ejecución falló')


## Parte A — carga viva

**Archivo:** `P1L3/results/transferencia_Q.csv`, generado por `P1L3/casos.py` en `vectors()`.

**En palabras simples:** se suman las cargas vivas transferidas y se comparan con `q_Q * area` para verificar que no se pierda carga.

In [ ]:
transfers = list(csv.DictReader((OUT/'transferencia_Q.csv').open(encoding='utf-8-sig')))
qA = sum(float(r['Q_kN']) for r in transfers)
q_area = sum(float(r['q_Q_kN_m2'])*float(r['area_m2']) for r in transfers)
print(f'Q transferida = {qA:.6f} kN')
print(f'q_Q · A       = {q_area:.6f} kN')
print(f'Error          = {qA-q_area:.3e} kN')


## Parte B — sismo pseudoestatico

**Archivo:** `P1L3/sismo.py`, funcion `calcular_pisos()`.

**En palabras simples:** se transforma el peso de cada piso en masa y luego en una fuerza horizontal para `EX` y `EY`.

In [ ]:
floors = list(csv.DictReader((OUT/'sismo_pisos.csv').open(encoding='utf-8-sig')))
for case in ['EX','EY']:
    rows = list(csv.DictReader((OUT/'sismo_por_piso.csv').open(encoding='utf-8-sig')))
    lateral = sum(abs(float(r['Fx_kN'] if case=='EX' else r['Fy_kN'])) for r in rows)
    print(case, 'fuerza lateral aplicada/reacción total ≈', lateral, 'kN')
print('Aceleración sísmica:', cfg['aceleracion_fraccion_g'], 'g')
print('Masa sísmica: G +', cfg['fraccion_Q_masa'], 'Q')


### Defensa: Q, EX y EY

- **Q** es la carga viva gravitacional de uso, en kN/m². Se transforma en fuerzas verticales nodales mediante `casos.vectors()` (archivo `P1L3/casos.py`), usando el área tributaria de cada receptor. Su conservación se controla en `P1L3/results/conservacion_Q.csv`: la suma transferida debe coincidir con `q_Q * A` o con la suma por zonas cuando `q_Q_kN_m2` es `null`.
- **EX** y **EY** son dos casos horizontales pseudoestáticos independientes. `sismo.calcular_pisos()` (archivo `P1L3/sismo.py`) calcula `W_i = G_i + fraccion_Q_masa Q_i`, `m_i = W_i/g` y `F_i = m_i a_i`. `casos.vectors()` aplica `F_i` en X para `EX` y en Y para `EY`, incluyendo el par de transporte al nodo maestro.
- La diferencia esencial es física y de dirección: Q es una acción vertical distribuida sobre losas; EX/EY son acciones inerciales laterales derivadas de la masa sísmica. Q puede participar en la masa sísmica, pero no se convierte por eso en una carga lateral directa.
- `casos.solve()` ejecuta cada caso por separado con `{case: 1}`. `casos.run()` guarda `Q_fuerzas*.json`, `EX_fuerzas*.json` y `EY_fuerzas*.json`; `ejecutar.py` arma los reportes. Para defenderlo, mostrar `transferencia_Q.csv`, `sismo_por_piso.csv` y `masas_y_sismo.csv` como trazabilidad.

**Pregunta de control:** si aumento `fraccion_Q_masa`, ¿qué cambia? Aumentan `W_i`, la masa y, por tanto, `F_i` en EX/EY; la carga gravitacional Q y su reparto no cambian.

## Parte C — superposicion

**Archivo:** `P1L3/casos.py`, funcion `solve()`, y `P1L3/parametros.json`.

**En palabras simples:** se multiplican las respuestas de los casos base por los factores de `R` y se comparan con una corrida explicita.

In [ ]:
summary = json.loads((OUT/'resumen_global.json').read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('La combinación R se contrasta con la corrida explícita en verificar_reparto_combinacion.py.')


## Parte D — capacidad HA

**Archivo:** `P1L3/capacidad.py`, funciones `fibers()`, `moment_curvature()` e `interaction_points()`.

**En palabras simples:** se arma una columna pequena con fibras de hormigon y acero, se le aplica carga axial y curvatura, y se obtiene su capacidad.

In [ ]:
pm = list(csv.DictReader((OUT/'PM_puntos.csv').open(encoding='utf-8-sig')))
for i, row in enumerate(pm, 1):
    print(f'Punto {i}: P={float(row["P_kN"]):.1f} kN, M={float(row["M_kNm"]):.1f} kN·m')
phi = list(csv.DictReader((OUT/'momento_curvatura.csv').open(encoding='utf-8-sig')))
fig, ax = plt.subplots(figsize=(7,4))
byP = {}
for r in phi: byP.setdefault(r['P_objetivo_kN'], []).append(r)
for p, rowsP in byP.items():
    ax.plot([float(r['phi_1_m']) for r in rowsP],[float(r['M_kNm']) for r in rowsP],label=f'P={float(p):.0f} kN')
ax.set_xlabel('Curvatura φ [1/m]'); ax.set_ylabel('Momento M [kN·m]'); ax.set_title('Momento–curvatura — Fiber Section'); ax.grid(alpha=.2); ax.legend(); plt.show()


## Defensa: Fiber Section y curva de capacidad

Una **Fiber Section** divide la sección transversal en pequeñas áreas llamadas fibras. Cada fibra conoce su posición, área y material: en `capacidad.fibers()` (archivo `P1L3/capacidad.py`) las fibras de hormigón reciben el material 1 (`Concrete01`) y las barras reciben el material 2 (`Steel01`). Se descuenta el área del acero de las celdas de hormigón para conservar `Ac + As = Ag`.

Su utilidad es representar la distribución no uniforme de deformaciones y tensiones en flexocompresión. `capacidad.moment_curvature()` crea la sección OpenSees con `ops.section('Fiber', 1)`, la usa en `zeroLengthSection`, aplica primero P y luego incrementa la curvatura con `DisplacementControl`. Al sumar las fuerzas de fibras se obtiene M para un P dado, es decir, una curva M-φ. La envolvente de esos estados a distintos P es la curva de capacidad P-M.

No debe confundirse con la sección elástica del edificio: la Fiber Section es el modelo no lineal separado de una columna HA. En este caso usa 0,70 × 0,70 m, `fc_MPa = 35`, `fy_MPa = 420`, `Concrete01` sin tracción y `Steel01` elastoplástico perfecto; los estribos se documentan, pero no agregan confinamiento constitutivo.

In [ ]:
# Curva P-M de capacidad: Fiber Section OpenSees vs envolvente independiente.
pm_fiber = list(csv.DictReader((OUT/'PM_puntos.csv').open(encoding='utf-8-sig')))
pm_ind = list(csv.DictReader((OUT/'PM_compatibilidad_envolvente_material.csv').open(encoding='utf-8-sig')))
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([float(r['M_kNm']) for r in pm_fiber], [float(r['P_kN']) for r in pm_fiber], 'o-', label='P-M nominal A-G')
ax.plot([float(r['M_kNm']) for r in pm_ind], [float(r['P_kN']) for r in pm_ind], 's--', label='Compatibilidad independiente')
for r in pm_fiber: ax.annotate(r['punto'], (float(r['M_kNm']), float(r['P_kN'])), xytext=(5, 4), textcoords='offset points')
ax.set_xlabel('Momento M [kN·m]'); ax.set_ylabel('Axial P [kN] (+ compresión)')
ax.set_title('Curva de capacidad P-M de la columna')
ax.grid(alpha=.2); ax.legend(); plt.show()
cap = json.loads((OUT/'resumen_capacidad.json').read_text(encoding='utf-8'))
print('Error de malla relativo:', cap['error_malla_relativo'])
print('Error axial máximo [kN]:', cap['error_axial_kN'])
print('Error de refinamiento de picos:', cap['error_pico_refinado_relativo'])


### Cómo se valida la curva

La línea `P-M nominal A-G` proviene de `capacidad.interaction_points()`: para cada estado se impone compatibilidad lineal de deformaciones, se calcula la tensión de cada capa y se integran fuerzas y momentos. La línea `Compatibilidad independiente` proviene de `capacidad.root_p()` y `capacidad.resultant()`, que integran las fibras con las leyes constitutivas; no reutiliza los puntos A-G.

La defensa debe comprobar: (1) equilibrio axial, comparando `P_objetivo_kN` con `P_seccion_kN` en `momento_curvatura.csv`; (2) independencia de discretización, comparando `fibras_por_lado = 40` con una malla doble; y (3) estabilidad numérica, comparando los picos con el paso de curvatura refinado. `resumen_capacidad.json` reporta `error_axial_kN`, `error_malla_relativo` y `error_pico_refinado_relativo`. Como chequeo manual rápido, en cada punto se usa `P = Σσ_i A_i` y `M = Σσ_i A_i z_i`; si la suma de áreas no es `0,49 m²`, la sección está mal construida.

## Flujo de la carga axial en columnas

En palabras simples, el recorrido es: geometria, pesos, reparto a nodos, solucion de OpenSees, extraccion del axial y comparacion con la capacidad. La demanda es lo que recibe la columna; la resistencia es lo que puede soportar.

**Archivo:** `P1L3/casos.py`, funcion `solve()`.

El codigo siguiente muestra la funcion real que resuelve el edificio.

In [ ]:
import inspect
sys.path.insert(0, str(ROOT))
import casos, capacidad
print(inspect.getsource(casos.solve))


**Archivo:** `P1L3/ejecutar.py`, bloque de trazabilidad vertical de pilares.

**En palabras simples:** se filtran las columnas, se busca que columnas estan encima y se muestra el axial de cada tramo.

In [ ]:
# La auditoria contiene el axial de cada columna y cada caso.
axiales = list(csv.DictReader((OUT/'auditoria_axiales_columnas.csv').open(encoding='utf-8-sig')))
for r in [r for r in axiales if r['caso'] == 'R'][:10]:
    print(f"Elemento {r['elemento']} | eje {r['eje']} | tramo {r['z_inferior_m']}-{r['z_superior_m']} m | P = {float(r['P_compresion_kN']):.3f} kN")


### Resistencia de la seccion

La seccion se divide en fibras de hormigon y acero. Para cada deformacion se calcula la tension de cada fibra, se suman las fuerzas y se obtienen `P` y `M`. Repitiendo el proceso para distintas posiciones del eje neutro se construye la curva `P-M`.

El siguiente bloque muestra la funcion real que genera sus puntos.

In [ ]:
print(inspect.getsource(capacidad.interaction_points))


## Preguntas conceptuales

### ¿Por que la superposicion funciona?

Funciona porque el modelo global es elastico y lineal. OpenSees resuelve `K u = F`: si la rigidez `K` no cambia, la respuesta producida por una suma de cargas es igual a la suma de las respuestas producidas por separado. Por eso se pueden sumar las respuestas de `G`, `Q`, `EX` y `EY` multiplicadas por sus factores.

### ¿Cuando dejaria de funcionar?

Dejaria de funcionar como suma exacta si la rigidez cambiara durante el analisis. Esto puede ocurrir por plastificacion, fisuracion dependiente de la carga, contacto, apertura de apoyos, grandes desplazamientos o efectos P-Delta importantes. En esos casos la respuesta debe obtenerse aplicando la combinacion directamente.

### ¿Que representa cada fibra?

Cada fibra representa una pequena porcion de la seccion transversal. Las fibras de hormigon tienen un area y una ley constitutiva de hormigon; las fibras de acero representan barras con su area, posicion y ley constitutiva del acero. Al sumar sus fuerzas se obtiene la respuesta completa de la seccion.

### ¿Por que `P` cambia la capacidad de `M`?

`P` es la fuerza axial y `M` es el momento. Al aplicar compresion, cambia la distribucion de deformaciones y esfuerzos entre el hormigon y el acero. Una compresion moderada puede aumentar el momento resistente, pero una compresion muy alta reduce la capacidad de deformacion y hace que el momento resistente tienda a cero. Por eso la capacidad se representa con una curva `P-M` y no con un unico momento maximo.

### ¿Que diferencia hay entre demanda y capacidad?

La **demanda** es la solicitacion que el edificio transmite a la columna: por ejemplo, `P` y `M` obtenidos desde OpenSees. La **capacidad** es la solicitacion maxima que la seccion armada puede resistir, calculada con sus dimensiones, materiales y enfierradura. La verificacion compara el punto de demanda con la curva de capacidad.

## Preguntas para la defensa individual

Las preguntas se pueden asignar por integrante. Cada respuesta debe mencionar la función, el archivo y un resultado verificable; no basta responder que el estado es `OK`.

**Integrante 1 — cargas y sismo (`casos.py` / `sismo.py`)**

1. ¿Cómo se obtiene Q desde `tributary_area_m2` y dónde se demuestra que no se pierde carga?
2. ¿Por qué Q participa con `fraccion_Q_masa`, pero EX y EY no son cargas verticales?
3. ¿Qué diferencia hay entre EX y EY y cómo se aplica el par de transporte al nodo maestro?

**Integrante 2 — solución y superposición (`casos.solve()` / `casos.run()`)**

1. ¿Por qué R se vuelve a resolver explícitamente y no se toma solo como suma de archivos?
2. ¿Qué tres familias de respuesta se comparan en `comparacion_superposicion.csv` y cuál es la tolerancia?
3. ¿Qué fenómeno haría inválida la suma lineal y qué análisis usaría en ese caso?

**Integrante 3 — capacidad (`capacidad.py`)**

1. ¿Qué representa una fibra y por qué se descuenta su área del hormigón?
2. ¿Qué hacen `moment_curvature()` e `interaction_points()` y por qué sus curvas no son exactamente la misma historia?
3. ¿Cómo se valida la curva además de mirarla: qué significan el equilibrio axial, el error de malla y el refinamiento?

## Reproducibilidad

Desde una terminal ubicada en `P1L3`, ejecutar `python ejecutar.py`. Los resultados se escriben en `P1L3/results`. Para abrir este archivo: `jupyter notebook notebooks/Semana3_LAB.ipynb`.